# 3단계: Q2 — 정상 학습 이미지를 줄이면 떨어지는 폭이 데이터셋마다 다른가

Q1 1위 설정을 데이터셋마다 모은 넷으로 학습 장수를 50장, 10장으로 줄여 돌린다. 전처리는 Q1과 같은 비교 단계(`Resize((256, 256))`, 자르지 않음)다.

| 방법 | 설정 | Q1에서 1위였던 곳 |
|---|---|---|
| PatchCore | WRN-50-2 (2,3) 10% | MVTec AD |
| PatchCore | WRN-50-2 (1,2,3) 1% | Real-IAD, VisA |
| PaDiM | WRN-50-2 (2,3), 차원 550 | MVTec AD, Real-IAD |
| PaDiM | WRN-50-2 (1,2,3), 차원 550 | VisA (차이 0.19점, 시드 표준편차 0.52·0.61) |

**돌리는 칸.** 설정 4 x 37 부류(MVTec 15, VisA 12, Real-IAD 10) x 시드 3 x 학습 장수 2(50, 10) = 888번. 전체 장수 칸은 Q1 결과(레포 `results/q1_*.csv`)를 그대로 쓴다.

**표본 고르기.** 시드마다 학습 목록에서 중복 없이 뽑는다(`data.subsample_train`). 같은 시드의 10장은 50장 안에 든다. 시드가 바꾸는 것은 고른 사진과 코어셋 시작점(PaDiM은 무작위 차원) 둘이라, 시드 표준편차에 둘이 섞여 있다. MVTec AD Toothbrush는 학습이 60장이라 50장이 전체의 83%다.

**PaDiM 공분산.** 위치마다 공분산을 사진 N장으로 추정하므로 계수가 N−1(50장이면 49, 10장이면 9)을 넘지 못하고 나머지 550−(N−1) 방향은 ε = 0.01만으로 채워진다(README 「Q1 설정」).

**시간.** Q1 기록으로 어림하면 6.8~8.6시간이다. 아래 끝은 예측·지표 시간만 더한 값(시험 사진 수는 그대로다), 위 끝은 학습 시간을 장수에 비례해 줄인 값이다. 코어셋 시간은 뱅크 크기의 제곱으로 줄어서 실제 학습 시간은 위 끝 계산보다 짧을 것으로 본다. 잰 값이 아니다.

**순서.** 시드 하나마다 MVTec -> Real-IAD -> VisA, 데이터셋마다 50장 -> 10장으로 돈다.

**끊겨도 된다.** 결과는 부류 하나가 끝날 때마다 Drive `beyond_mvtec_results/q2_*.csv`에 적는다. 런타임이 끊기면 1·2번 셀과 4번의 첫 셀을 다시 돌린 뒤 5번 셀을 다시 돌리면 끝난 줄을 건너뛴다.

**실행 전 준비.** 런타임 GPU(L4), 보안 비밀 `GITHUB_TOKEN` 접근 허용. `configs/experiments.yaml`의 `few_shot.settings`가 GitHub main에 올라가 있어야 1번 셀을 넘어간다.

## 1. 코드와 패키지

In [1]:
import os, subprocess, sys

REPO_DIR = "/content/beyond-mvtec"
REPO_URL = "https://github.com/chlee1027/beyond-mvtec.git"
UPSTREAM_DIR = "/content/patchcore-inspection"
UPSTREAM_COMMIT = "fcaa92f124fb1ad74a7acf56726decd4b27cbcad"

from google.colab import userdata
token = userdata.get("GITHUB_TOKEN").strip()
auth_url = REPO_URL.replace("https://", f"https://x-access-token:{token}@")
if not os.path.isdir(REPO_DIR):
    r = subprocess.run(["git", "clone", "-q", auth_url, REPO_DIR], capture_output=True, text=True)
else:
    # 이미 받아 둔 레포는 GitHub 의 최신 main 으로 맞춘다(토큰 든 주소는 저장하지 않는다)
    r = subprocess.run(["git", "-C", REPO_DIR, "pull", "-q", "--ff-only", auth_url, "main"],
                       capture_output=True, text=True)
# 실패해도 오류 메시지에 토큰이 찍히지 않게 가린다
assert r.returncode == 0, "레포 받기 실패: " + r.stderr.replace(token, "***")
subprocess.run(["git", "-C", REPO_DIR, "remote", "set-url", "origin", REPO_URL], check=True)
if not os.path.isdir(UPSTREAM_DIR):
    subprocess.run(["git", "clone", "-q", "https://github.com/amazon-science/patchcore-inspection.git", UPSTREAM_DIR], check=True)
subprocess.run(["git", "-C", UPSTREAM_DIR, "checkout", "-q", UPSTREAM_COMMIT], check=True)

!pip uninstall -y -q sympy
!pip install -q sympy==1.13.3 faiss-gpu-cu12 timm click pretrainedmodels scikit-image scikit-learn scipy tqdm pandas pyyaml

sys.path[:0] = [os.path.join(REPO_DIR, "src"), os.path.join(UPSTREAM_DIR, "src")]
COMMIT = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()

import faiss, torch, torchvision
assert torch.cuda.is_available(), "GPU 런타임이 아니다"
print(f"레포 {COMMIT} | Python {sys.version.split()[0]} | torch {torch.__version__} | torchvision {torchvision.__version__} | faiss {faiss.__version__}")
print(f"GPU {torch.cuda.get_device_name(0)} | VRAM {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
assert hasattr(faiss, "StandardGpuResources"), "faiss 의 GPU 판이 아니다"

from beyond_mvtec import data, experiment as ex, patchcore_adapter
config = ex.load_config(f"{REPO_DIR}/configs/experiments.yaml")
Q2 = config.get("few_shot", {})
# 레포가 Q2 설정을 적기 전 판이면 여기서 멈춘다
assert "settings" in Q2 and hasattr(patchcore_adapter, "_releasing_sampler") \
    and config["datasets"]["real_iad"].get("split") == "sv", \
    "옛 코드가 읽혔다. Q2 설정을 push 했는지 확인하고, 런타임 > 세션 다시 시작 후 1번 셀부터 다시 돌린다"
SETTINGS = [s for s in ex.comparison_settings(config) if s["setting"] in Q2["settings"]]
assert sorted(s["setting"] for s in SETTINGS) == sorted(Q2["settings"]) and len(SETTINGS) == 4, Q2["settings"]
N_TRAIN = [int(n) for n in Q2["n_train"] if str(n) != "all"]
assert N_TRAIN == [50, 10], N_TRAIN
print("설정", len(SETTINGS), "개:", [s["setting"] for s in SETTINGS], "| 학습 장수", N_TRAIN, "(전체는 Q1 결과)")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
레포 dd716bc | Python 3.13.16 | torch 2.14.1+cu130 | torchvision 0.29.1+cu130 | faiss 1.14.1
GPU NVIDIA L4 | VRAM 23.7 GB
설정 4 개: ['patchcore_wrn50_L23_c10', 'patchcore_wrn50_L123_c1', 'padim_wrn50_L123_d550', 'padim_wrn50_L23_d550'] | 학습 장수 [50, 10] (전체는 Q1 결과)


## 2. Drive와 데이터

In [2]:
import shutil, zipfile
from google.colab import drive
drive.mount("/content/drive")

MY = "/content/drive/MyDrive"
os.environ["TORCH_HOME"] = f"{MY}/torch_cache"
RESULT_DIR = f"{MY}/beyond_mvtec_results"
os.makedirs(RESULT_DIR, exist_ok=True)

def copy_once(src, dst):
    """Drive 에서 파일을 하나씩 읽으면 느려서 런타임 디스크로 한 번 옮긴다. 진행 표시는 없다."""
    if not os.path.isfile(f"{dst}/.copied"):
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        shutil.copytree(src, dst, dirs_exist_ok=True)
        open(f"{dst}/.copied", "w").close()
    print("Drive:", src, "-> 로컬:", dst)

MVTEC_DRIVE = next((p for p in [f"{MY}/anormaly_detection/mvtec", f"{MY}/mvtec", f"{MY}/MVTecAD"]
                    if os.path.isdir(f"{p}/bottle")), None)
VISA_DRIVE = next((p for p in [f"{MY}/visa", f"{MY}/VisA_20220922", f"{MY}/VisA"]
                   if os.path.isfile(f"{p}/split_csv/1cls.csv")), None)
assert MVTEC_DRIVE and VISA_DRIVE, (MVTEC_DRIVE, VISA_DRIVE)

ROOTS = {"mvtec_ad": "/content/data/mvtec_ad", "visa": "/content/data/visa", "real_iad": "/content/data/real_iad"}
copy_once(MVTEC_DRIVE, ROOTS["mvtec_ad"])
copy_once(VISA_DRIVE, ROOTS["visa"])

# Real-IAD: 분할 JSON 을 풀고, 부류 zip 에서 C1 사진과 마스크만 푼다
RI = config["datasets"]["real_iad"]
VIEW, RI_SPLIT = RI["views"][0], RI["split"]
os.makedirs(ROOTS["real_iad"], exist_ok=True)
if not os.path.isdir(f"{ROOTS['real_iad']}/realiad_jsons"):
    with zipfile.ZipFile(f"{MY}/realiad_jsons.zip") as z:
        z.extractall(f"{ROOTS['real_iad']}/realiad_jsons")
for category in RI["categories"]:
    data.prepare_real_iad(f"{MY}/Real_IAD/{category}.zip", ROOTS["real_iad"], category, view=VIEW)
print("Real-IAD", len(RI["categories"]), "부류:", RI["categories"], "| 시점", VIEW, "| 분할", RI_SPLIT)

CATEGORIES = {"mvtec_ad": data.MVTEC_CATEGORIES, "visa": data.visa_categories(ROOTS["visa"]),
              "real_iad": RI["categories"]}

Mounted at /content/drive
Drive: /content/drive/MyDrive/mvtec -> 로컬: /content/data/mvtec_ad
Drive: /content/drive/MyDrive/visa -> 로컬: /content/data/visa
Real-IAD 10 부류: ['bottle_cap', 'pcb', 'phone_battery', 'porcelain_doll', 'regulator', 'sim_card_set', 'switch', 'usb', 'wooden_beads', 'woodstick'] | 시점 C1 | 분할 sv


In [3]:
import pandas as pd

rows = []
for dataset, categories in CATEGORIES.items():
    for category in categories:
        split = ex.list_split(dataset, ROOTS[dataset], category, VIEW, real_iad_split=RI_SPLIT)
        rows.append({"dataset": dataset, "category": category, "train": len(split.train),
                     "test": len(split.test), "test_anomaly": sum(s.label for s in split.test)})
counts = pd.DataFrame(rows)
display(counts.groupby("dataset")[["train", "test", "test_anomaly"]].sum())

# 00 노트북에서 센 장수와 같아야 한다
total = counts.groupby("dataset")[["train", "test"]].sum()
assert tuple(total.loc["mvtec_ad"]) == (3629, 1725), tuple(total.loc["mvtec_ad"])
assert tuple(total.loc["visa"]) == (8659, 2162), tuple(total.loc["visa"])
expected_ri = {"bottle_cap": (370, 633), "pcb": (279, 723), "phone_battery": (350, 650), "porcelain_doll": (402, 598),
               "regulator": (477, 543), "sim_card_set": (305, 701), "switch": (267, 743), "usb": (354, 656),
               "wooden_beads": (305, 696), "woodstick": (443, 558)}
got_ri = {r.category: (r.train, r.test) for r in counts[counts.dataset == "real_iad"].itertuples()}
assert got_ri == expected_ri, got_ri
print("장수가 00 노트북과 같다")
display(counts[counts.dataset == "real_iad"])

,train,test,test_anomaly
dataset,,,
mvtec_ad,3629,1725,1258
real_iad,3552,6501,2956
visa,8659,2162,1200


장수가 00 노트북과 같다


,dataset,category,train,test,test_anomaly
27,real_iad,bottle_cap,370,633,264
28,real_iad,pcb,279,723,445
29,real_iad,phone_battery,350,650,301
30,real_iad,porcelain_doll,402,598,196
31,real_iad,regulator,477,543,66
32,real_iad,sim_card_set,305,701,396
33,real_iad,switch,267,743,477
34,real_iad,usb,354,656,303
35,real_iad,wooden_beads,305,696,392
36,real_iad,woodstick,443,558,116


## 3. 전체 장수 칸(Q1 결과) 확인

레포 `results/`의 Q1 CSV에서 이 노트북의 설정 넷만 꺼낸다. 칸이 모자라면 멈춘다.

In [4]:
Q1_ROWS = pd.concat([pd.read_csv(f"{REPO_DIR}/results/q1_{d}.csv") for d in ROOTS], ignore_index=True)
Q1_ROWS = Q1_ROWS[Q1_ROWS.setting.isin(Q2["settings"]) & (Q1_ROWS.n_train_requested.astype(str) == "all")]
for dataset, categories in CATEGORIES.items():
    part = Q1_ROWS[Q1_ROWS.dataset == dataset]
    assert len(part) == len(SETTINGS) * len(categories) * len(config["seeds"]), (dataset, len(part))
    assert set(part.category) == set(categories), dataset
print("Q1 전체 장수 칸", len(Q1_ROWS), "줄 | 커밋", sorted(Q1_ROWS.commit.astype(str).unique()))

Q1 전체 장수 칸 444 줄 | 커밋 ['88fd4dc']


## 4. 짧은 실행: 표본 확인과 10장 한 부류

먼저 37 부류 x 시드 3에서 10장이 50장 안에 드는지, 고른 장수가 맞는지 센다(학습 없음). 그다음 VisA `pcb3` 시드 0의 10장으로 설정 넷을 돌린다. 결과는 5번과 같은 CSV에 적히므로 5번에서 다시 돌지 않는다.

**넘어가는 기준.** 네 줄 모두 오류 없이 끝나고 `n_train`이 10이면 5번으로 간다. 오류가 나면 5번을 돌리지 말고 이 셀 출력을 보낸다.

In [5]:
PREPROCESS = config["preprocess"]["compare"]
OPTIONS = dict(device="cuda", batch_size=8, num_workers=4, faiss_on_gpu=True, commit=COMMIT,
               view=VIEW, real_iad_split=RI_SPLIT, save_dir=f"{RESULT_DIR}/q2_scores")
CSV = {d: f"{RESULT_DIR}/q2_{d}.csv" for d in ROOTS}

In [6]:
for dataset, categories in CATEGORIES.items():
    for category in categories:
        train = ex.list_split(dataset, ROOTS[dataset], category, VIEW, real_iad_split=RI_SPLIT).train
        for seed in config["seeds"]:
            picked = {n: data.subsample_train(train, n, seed) for n in N_TRAIN}
            assert all(len(picked[n]) == n for n in N_TRAIN), (dataset, category, seed)
            assert set(picked[10]) <= set(picked[50]), (dataset, category, seed)
print("37 부류 x 시드 3: 장수가 맞고 10장이 50장 안에 든다")

ex.run_grid(SETTINGS, "visa", ROOTS["visa"], ["pcb3"], [0], "compare", PREPROCESS, CSV["visa"],
            n_train=10, **OPTIONS)
done = pd.read_csv(CSV["visa"])
display(done[(done.category == "pcb3") & (done.seed == 0) & (done.n_train_requested.astype(str) == "10")]
        [["setting", "n_train", "image_auroc", "pixel_auroc", "pro", "bank_size", "fit_sec", "predict_sec",
          "metric_sec", "peak_gpu_gb"]])

37 부류 x 시드 3: 장수가 맞고 10장이 50장 안에 든다


,setting,n_train,image_auroc,pixel_auroc,pro,bank_size,fit_sec,predict_sec,metric_sec,peak_gpu_gb
0,patchcore_wrn50_L23_c10,10,0.868119,0.986112,0.836805,1024.0,1.68,7.92,7.20,1.22
1,patchcore_wrn50_L123_c1,10,0.907921,0.950210,0.743861,409.0,1.94,25.53,7.64,4.58
2,padim_wrn50_L123_d550,10,0.719703,0.978789,0.801349,NaN,12.55,2.10,7.30,17.72
3,padim_wrn50_L23_d550,10,0.727129,0.978857,0.769756,NaN,1.83,1.54,7.33,6.51


## 5. Q2 실행

점수와 라벨은 `q2_scores/`에 남기고 이상 지도는 남기지 않는다.

In [7]:
for seed in config["seeds"]:
    for dataset in ["mvtec_ad", "real_iad", "visa"]:
        for n_train in N_TRAIN:
            print(f"===== 시드 {seed} | {dataset} | 학습 {n_train}장 =====")
            ex.run_grid(SETTINGS, dataset, ROOTS[dataset], CATEGORIES[dataset], [seed], "compare", PREPROCESS,
                        CSV[dataset], n_train=n_train, **OPTIONS)

===== 시드 0 | mvtec_ad | 학습 50장 =====
===== 시드 0 | mvtec_ad | 학습 10장 =====
===== 시드 0 | real_iad | 학습 50장 =====
===== 시드 0 | real_iad | 학습 10장 =====
===== 시드 0 | visa | 학습 50장 =====
===== 시드 0 | visa | 학습 10장 =====
===== 시드 1 | mvtec_ad | 학습 50장 =====
===== 시드 1 | mvtec_ad | 학습 10장 =====
===== 시드 1 | real_iad | 학습 50장 =====
===== 시드 1 | real_iad | 학습 10장 =====
===== 시드 1 | visa | 학습 50장 =====
===== 시드 1 | visa | 학습 10장 =====
===== 시드 2 | mvtec_ad | 학습 50장 =====
===== 시드 2 | mvtec_ad | 학습 10장 =====
===== 시드 2 | real_iad | 학습 50장 =====
===== 시드 2 | real_iad | 학습 10장 =====


/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:207: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:222: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=Wide_ResNet50_2_Weights.IMAGENET1K_V1`. You can also use `weights=Wide_ResNet50_2_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)

Subsampling...: 100%|██████████| 409/409 [00:00<00:00, 2255.22it/s]


[16/40] patchcore_wrn50_L123_c1 sim_card_set s2 I=0.9634 P=0.9973 PRO=0.9785 fit=2.44s pred=88.24s gpu=4.58GB


/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:207: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:222: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=Wide_ResNet50_2_Weights.IMAGENET1K_V1`. You can also use `weights=Wide_ResNet50_2_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)

Subsampling...: 100%|██████████| 409/409 [00:00<00:00, 2982.56it/s]


[17/40] patchcore_wrn50_L123_c1 switch s2 I=0.8301 P=0.9673 PRO=0.9164 fit=2.32s pred=94.63s gpu=4.58GB


/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:207: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:222: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=Wide_ResNet50_2_Weights.IMAGENET1K_V1`. You can also use `weights=Wide_ResNet50_2_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)

Subsampling...: 100%|██████████| 409/409 [00:00<00:00, 3007.07it/s]


... (97 outputs left out)

/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:207: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:222: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=Wide_ResNet50_2_Weights.IMAGENET1K_V1`. You can also use `weights=Wide_ResNet50_2_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)

Subsampling...: 100%|██████████| 409/409 [00:00<00:00, 2883.00it/s]


[23/48] patchcore_wrn50_L123_c1 pcb4 s2 I=0.9715 P=0.8631 PRO=0.5553 fit=2.44s pred=25.44s gpu=4.58GB


/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:207: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:222: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=Wide_ResNet50_2_Weights.IMAGENET1K_V1`. You can also use `weights=Wide_ResNet50_2_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)

Subsampling...: 100%|██████████| 409/409 [00:00<00:00, 2618.48it/s]


[24/48] patchcore_wrn50_L123_c1 pipe_fryum s2 I=0.9872 P=0.9910 PRO=0.9557 fit=2.43s pred=19.33s gpu=4.58GB
[25/48] padim_wrn50_L123_d550 candle s2 I=0.8711 P=0.9821 PRO=0.9515 fit=7.72s pred=2.12s gpu=17.72GB
[26/48] padim_wrn50_L123_d550 capsules s2 I=0.5668 P=0.8770 PRO=0.5433 fit=7.51s pred=1.77s gpu=17.72GB
[27/48] padim_wrn50_L123_d550 cashew s2 I=0.7186 P=0.9764 PRO=0.8502 fit=7.66s pred=1.71s gpu=17.72GB
[28/48] padim_wrn50_L123_d550 chewinggum s2 I=0.9022 P=0.9810 PRO=0.6932 fit=7.62s pred=1.7s gpu=17.72GB
[29/48] padim_wrn50_L123_d550 fryum s2 I=0.7600 P=0.9526 PRO=0.7580 fit=7.68s pred=1.7s gpu=17.72GB
[30/48] padim_wrn50_L123_d550 macaroni1 s2 I=0.6697 P=0.9726 PRO=0.8302 fit=7.66s pred=2.11s gpu=17.72GB
[31/48] padim_wrn50_L123_d550 macaroni2 s2 I=0.6141 P=0.9465 PRO=0.7088 fit=7.53s pred=2.14s gpu=17.72GB
[32/48] padim_wrn50_L123_d550 pcb1 s2 I=0.7677 P=0.9128 PRO=0.6803 fit=7.7s pred=2.11s gpu=17.72GB
[33/48] padim_wrn50_L123_d550 pcb2 s2 I=0.7943 P=0.9782 PRO=0.8283 fit

## 6. 결과

Q1 전체 장수 칸과 합쳐 시드마다 부류 평균을 내고, 시드 셋의 평균 ± 표준편차(n−1)를 적는다. 단위는 %다.

- **값 표**: 설정 x (데이터셋, 학습 장수).
- **떨어진 폭**: 시드마다 (전체 − N장)을 낸 뒤 평균 ± 표준편차. 같은 시드끼리 뺀다.
- **순위**: (데이터셋, 학습 장수)마다 설정 넷의 순위(1 = 가장 높음). Q1 1위 설정이 장수를 줄여도 1위인지 본다.

시드 0만 끝난 상태에서도 돌릴 수 있다(표준편차는 비어 나온다).

In [8]:
q2_paths = [p for p in CSV.values() if os.path.isfile(p)]
rows = pd.concat([Q1_ROWS] + [pd.read_csv(p) for p in q2_paths], ignore_index=True)
rows["n_train_requested"] = rows.n_train_requested.astype(str)
ORDER = ["all"] + [str(n) for n in N_TRAIN]
METRICS = ["image_auroc", "pixel_auroc", "pro"]

per_seed = rows.groupby(["dataset", "setting", "n_train_requested", "seed"]).agg(
    **{m: (m, "mean") for m in METRICS}, n_categories=("category", "nunique")).reset_index()
per_seed[METRICS] *= 100
expected_n = {d: len(c) for d, c in CATEGORIES.items()}
cells = per_seed.groupby(["dataset", "setting", "n_train_requested"]).agg(
    n_seeds=("seed", "nunique"), n_categories=("n_categories", "min")).reset_index()
complete = len(cells) == len(SETTINGS) * len(ORDER) * len(CATEGORIES) and all(
    (r.n_seeds == 3) and (r.n_categories == expected_n[r.dataset]) for r in cells.itertuples())
print("칸이 다 찼나:", complete)

def mean_std(frame, value):
    g = frame.groupby(["setting", "dataset", "n_train_requested"])[value]
    out = g.mean().round(2).astype(str) + " ± " + g.std(ddof=1).round(2).astype(str)
    return out.unstack(["dataset", "n_train_requested"]).reindex(
        columns=pd.MultiIndex.from_product([list(CATEGORIES), ORDER]))

for m in METRICS:
    print(m, "(%, 시드 평균 ± 표준편차)")
    display(mean_std(per_seed, m))

칸이 다 찼나: True
image_auroc (%, 시드 평균 ± 표준편차)


mvtec_ad                              \
                                  all            50            10   
setting                                                             
padim_wrn50_L123_d550    95.15 ± 0.99  90.24 ± 0.79  84.24 ± 1.67   
padim_wrn50_L23_d550     95.47 ± 0.22   90.8 ± 0.95   83.9 ± 2.28   
patchcore_wrn50_L123_c1   98.2 ± 0.05  95.96 ± 0.24   90.3 ± 0.49   
patchcore_wrn50_L23_c10  98.55 ± 0.04  96.83 ± 0.41  92.95 ± 0.46   

                                 visa                              \
                                  all            50            10   
setting                                                             
padim_wrn50_L123_d550    87.96 ± 0.52   83.35 ± 1.4  76.45 ± 1.28   
padim_wrn50_L23_d550     87.77 ± 0.61  84.12 ± 1.58  78.09 ± 0.28   
patchcore_wrn50_L123_c1   96.6 ± 0.09  93.47 ± 0.22  87.48 ± 1.11   
patchcore_wrn50_L23_c10  93.14 ± 0.25  90.33 ± 0.14  86.27 ± 0.86   

                             real_iad                              
                                  all            50            10  
setting                                                            
padim_wrn50_L123_d550    88.03 ± 0.42  81.23 ± 0.59  72.45 ± 0.43  
padim_wrn50_L23_d550     88.67 ± 0.26  82.69 ± 0.15  74.17 ± 0.72  
patchcore_wrn50_L123_c1  93.55 ± 0.06  91.44 ± 0.19  84.46 ± 0.44  
patchcore_wrn50_L23_c10  92.56 ± 0.03  88.98 ± 0.18  83.87 ± 0.37

pixel_auroc (%, 시드 평균 ± 표준편차)


mvtec_ad                              \
                                  all            50            10   
setting                                                             
padim_wrn50_L123_d550    97.76 ± 0.07   97.0 ± 0.12  95.02 ± 0.13   
padim_wrn50_L23_d550     97.72 ± 0.13  97.07 ± 0.18  95.26 ± 0.31   
patchcore_wrn50_L123_c1  98.06 ± 0.03  97.37 ± 0.04  95.53 ± 0.32   
patchcore_wrn50_L23_c10   98.38 ± 0.0  98.02 ± 0.06   96.48 ± 0.1   

                                 visa                              \
                                  all            50            10   
setting                                                             
padim_wrn50_L123_d550    98.27 ± 0.04  97.42 ± 0.09   95.8 ± 0.31   
padim_wrn50_L23_d550     98.09 ± 0.01  97.25 ± 0.08    95.7 ± 0.3   
patchcore_wrn50_L123_c1  98.71 ± 0.05  97.87 ± 0.13   95.9 ± 0.49   
patchcore_wrn50_L23_c10  98.63 ± 0.03   98.21 ± 0.1  97.61 ± 0.19   

                             real_iad                              
                                  all            50            10  
setting                                                            
padim_wrn50_L123_d550    98.66 ± 0.04  98.08 ± 0.06  96.35 ± 0.15  
padim_wrn50_L23_d550     98.54 ± 0.05  97.98 ± 0.13  96.27 ± 0.13  
patchcore_wrn50_L123_c1  99.39 ± 0.04  99.15 ± 0.01  97.65 ± 0.97  
patchcore_wrn50_L23_c10  99.08 ± 0.01   98.8 ± 0.08  97.94 ± 0.14

pro (%, 시드 평균 ± 표준편차)


mvtec_ad                              \
                                  all            50            10   
setting                                                             
padim_wrn50_L123_d550    93.88 ± 0.17  92.16 ± 0.25  87.95 ± 0.33   
padim_wrn50_L23_d550      93.7 ± 0.23   92.1 ± 0.29  87.99 ± 0.71   
patchcore_wrn50_L123_c1  94.53 ± 0.02  93.28 ± 0.22   88.96 ± 0.7   
patchcore_wrn50_L23_c10  94.43 ± 0.03  93.23 ± 0.26   89.2 ± 0.15   

                                 visa                              \
                                  all            50            10   
setting                                                             
padim_wrn50_L123_d550    86.61 ± 0.24  82.97 ± 0.44  77.46 ± 0.58   
padim_wrn50_L23_d550      85.36 ± 0.2  82.11 ± 0.22  77.41 ± 0.45   
patchcore_wrn50_L123_c1  92.69 ± 0.21  88.92 ± 0.16  80.92 ± 1.47   
patchcore_wrn50_L23_c10   90.6 ± 0.15  87.85 ± 0.47  84.77 ± 0.64   

                             real_iad                              
                                  all            50            10  
setting                                                            
padim_wrn50_L123_d550    92.63 ± 0.16  89.99 ± 0.04  83.13 ± 0.56  
padim_wrn50_L23_d550     91.85 ± 0.08  89.28 ± 0.19   82.48 ± 0.6  
patchcore_wrn50_L123_c1  95.98 ± 0.14  94.64 ± 0.22  88.96 ± 2.56  
patchcore_wrn50_L23_c10   93.5 ± 0.03  91.48 ± 0.24   87.2 ± 0.58

In [9]:
# 떨어진 폭: 같은 시드의 전체 장수 값에서 뺀다
full = per_seed[per_seed.n_train_requested == "all"].set_index(["dataset", "setting", "seed"])[METRICS]
drop = per_seed[per_seed.n_train_requested != "all"].join(full, on=["dataset", "setting", "seed"], rsuffix="_all")
for m in METRICS:
    drop[f"{m}_drop"] = drop[f"{m}_all"] - drop[m]
for m in ["image_auroc"]:
    print("이미지 AUROC 떨어진 폭 (전체 − N장, %p, 시드 평균 ± 표준편차)")
    g = drop.groupby(["setting", "dataset", "n_train_requested"])[f"{m}_drop"]
    display((g.mean().round(2).astype(str) + " ± " + g.std(ddof=1).round(2).astype(str))
            .unstack(["dataset", "n_train_requested"])
            .reindex(columns=pd.MultiIndex.from_product([list(CATEGORIES), ORDER[1:]])))

# 순위: (데이터셋, 장수)마다 설정 넷
values = per_seed.groupby(["setting", "dataset", "n_train_requested"]).image_auroc.mean().unstack(
    ["dataset", "n_train_requested"]).reindex(columns=pd.MultiIndex.from_product([list(CATEGORIES), ORDER]))
print("이미지 AUROC 순위 (1 = 가장 높음, 설정 넷 안에서)")
display(values.rank(ascending=False, method="average"))

이미지 AUROC 떨어진 폭 (전체 − N장, %p, 시드 평균 ± 표준편차)


mvtec_ad                       visa                \
                                  50            10           50            10   
setting                                                                         
padim_wrn50_L123_d550    4.91 ± 0.24  10.91 ± 1.06   4.61 ± 0.9  11.51 ± 0.92   
padim_wrn50_L23_d550     4.67 ± 0.77  11.57 ± 2.13  3.65 ± 0.99   9.68 ± 0.49   
patchcore_wrn50_L123_c1  2.24 ± 0.25    7.9 ± 0.44  3.13 ± 0.14   9.12 ± 1.05   
patchcore_wrn50_L23_c10  1.72 ± 0.43   5.59 ± 0.48  2.81 ± 0.27   6.87 ± 1.09   

                            real_iad                
                                  50            10  
setting                                             
padim_wrn50_L123_d550    6.81 ± 0.22  15.59 ± 0.45  
padim_wrn50_L23_d550     5.98 ± 0.26   14.5 ± 0.97  
patchcore_wrn50_L123_c1   2.11 ± 0.2   9.08 ± 0.47  
patchcore_wrn50_L23_c10  3.58 ± 0.21   8.69 ± 0.35

이미지 AUROC 순위 (1 = 가장 높음, 설정 넷 안에서)


mvtec_ad           visa           real_iad          
                             all   50   10  all   50   10      all   50   10
setting                                                                     
padim_wrn50_L123_d550        4.0  4.0  3.0  3.0  4.0  4.0      4.0  4.0  4.0
padim_wrn50_L23_d550         3.0  3.0  4.0  4.0  3.0  3.0      3.0  3.0  3.0
patchcore_wrn50_L123_c1      2.0  2.0  2.0  1.0  1.0  1.0      1.0  1.0  1.0
patchcore_wrn50_L23_c10      1.0  1.0  1.0  2.0  2.0  2.0      2.0  2.0  2.0

In [10]:
# 시간과 메모리: 설정 x 데이터셋 x 장수, 부류 한 번 평균 (Q2 실행분만)
q2 = pd.concat([pd.read_csv(p) for p in q2_paths], ignore_index=True)
display(q2.groupby(["dataset", "setting", "n_train_requested"])[
    ["n_train", "bank_size", "fit_sec", "predict_sec", "metric_sec", "peak_gpu_gb"]].mean().round(1))
print("더한 시간(시간):", round((q2.fit_sec + q2.predict_sec + q2.metric_sec).sum() / 3600, 1))

n_train  bank_size  \
dataset  setting                 n_train_requested                       
mvtec_ad padim_wrn50_L123_d550   10                    10.0        NaN   
                                 50                    50.0        NaN   
         padim_wrn50_L23_d550    10                    10.0        NaN   
                                 50                    50.0        NaN   
         patchcore_wrn50_L123_c1 10                    10.0      409.0   
                                 50                    50.0     2048.0   
         patchcore_wrn50_L23_c10 10                    10.0     1024.0   
                                 50                    50.0     5120.0   
real_iad padim_wrn50_L123_d550   10                    10.0        NaN   
                                 50                    50.0        NaN   
         padim_wrn50_L23_d550    10                    10.0        NaN   
                                 50                    50.0        NaN   
         patchcore_wrn50_L123_c1 10                    10.0      409.0   
                                 50                    50.0     2048.0   
         patchcore_wrn50_L23_c10 10                    10.0     1024.0   
                                 50                    50.0     5120.0   
visa     padim_wrn50_L123_d550   10                    10.0        NaN   
                                 50                    50.0        NaN   
         padim_wrn50_L23_d550    10                    10.0        NaN   
                                 50                    50.0        NaN   
         patchcore_wrn50_L123_c1 10                    10.0      409.0   
                                 50                    50.0     2048.0   
         patchcore_wrn50_L23_c10 10                    10.0     1024.0   
                                 50                    50.0     5120.0   

                                                    fit_sec  predict_sec  \
dataset  setting                 n_train_requested                         
mvtec_ad padim_wrn50_L123_d550   10                     7.8          1.8   
                                 50                     9.1          1.8   
         padim_wrn50_L23_d550    10                     2.1          1.6   
                                 50                     2.5          1.6   
         patchcore_wrn50_L123_c1 10                     2.6         15.2   
                                 50                    11.9         15.6   
         patchcore_wrn50_L23_c10 10                     1.5          4.8   
                                 50                     4.7          4.9   
real_iad padim_wrn50_L123_d550   10                     7.8          5.9   
                                 50                     8.9          6.0   
         padim_wrn50_L23_d550    10                     1.9          4.2   
                                 50                     2.3          4.2   
         patchcore_wrn50_L123_c1 10                     2.4         80.2   
                                 50                    11.7         81.0   
         patchcore_wrn50_L23_c10 10                     1.3         23.9   
                                 50                     4.6         24.5   
visa     padim_wrn50_L123_d550   10                     7.8          2.0   
                                 50                     8.9          1.9   
         padim_wrn50_L23_d550    10                     1.9          1.5   
                                 50                     2.3          1.5   
         patchcore_wrn50_L123_c1 10                     2.4         22.8   
                                 50                    11.6         23.4   
         patchcore_wrn50_L23_c10 10                     1.3          6.9   
                                 50                     4.6          7.4   

                                                    metric_sec  peak_gpu_gb  
dataset  setting                 n_train_requested                      

더한 시간(시간): 7.3


## 7. 결과를 레포로

Drive `beyond_mvtec_results/`의 `q2_mvtec_ad.csv`, `q2_real_iad.csv`, `q2_visa.csv`와 실행한 이 노트북을 바탕화면에 내려받는다. `q2_scores/`는 올리지 않는다.